# 03 - HAAQI Score
Compute HAAQI score for the baseline output 
Run `01_setup.ipynb` first (needs `clarity/` cloned and installed).

##### Needs about 30 GB of free disk and a long first 12.6 GB download

1. Download the valid-set data i.e. three files from Zenodo

In [1]:
import hashlib, subprocess
from pathlib import Path

ZENODO = 'https://zenodo.org/api/records/14335552/files'
FILES = {
    'metadata.zip': '7ff49002ae62023cd547ff057cbba461',
    'EnsembleSet_Mix_1.zip': '5bdcd0154da69e7516aef2d3062a47f0',
    'CadenzaWoodwind.zip': 'cdc08be8482662c5023b5d4060d1b064',
}
dl = Path('zenodo_download').resolve()
dl.mkdir(exist_ok=True)

def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

for name, want in FILES.items():
    path = dl / name
    if not path.exists() or md5sum(path) != want:
        subprocess.run(['wget', '-q', '-c', '-O', str(path), f'{ZENODO}/{name}/content'], check=True)
    assert md5sum(path) == want, f'md5 mismatch for {name}'
    print('OK', name)

OK metadata.zip
OK EnsembleSet_Mix_1.zip
OK CadenzaWoodwind.zip


2. Build the dataset. Clarity's script unzips the files and creates the `audio/` and `metadata/` folders

In [2]:
root = Path('cadenza_data/cad2/task2').resolve()
clarity_dir = Path('clarity').resolve()

# Colab's torchtune has its own `recipes` package; PYTHONPATH makes Clarity's win.
!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/process_dataset/process_zenodo_download.py \
    path.zenodo_download_path={dl} \
    path.root={root}

Extracting CadenzaWoodwind: 100% 172/172 [01:39<00:00,  1.73it/s]
Extracting EnsembleSet_Mix_1: 100% 548/548 [00:54<00:00, 10.00it/s]
Extracting metadata: 100% 18/18 [00:00<00:00, 702.36it/s]
Copying training audio: 100% 104/104 [01:46<00:00,  1.03s/it]
Creating validation audio: 100% 261/261 [01:38<00:00,  2.64it/s]
Copying metadata directory.


In [3]:
meta = root / 'metadata'
for name in ['scenes.valid.json', 'scene_listeners.valid.json', 'music.valid.json', 'listeners.valid.json']:
    assert (meta / name).exists(), f'missing {name}'
print('valid audio folders:', sorted(p.name for p in (root / 'audio' / 'valid').iterdir())[:5], '...')

valid audio folders: ['Calcante_ed_Achille_001', 'Marche_001', 'Marche_002', 'Precipitoso_nel_mar_001', 'Precipitoso_nel_mar_002'] ...


3. Pick a first scene in the valid set with its first listener

In [4]:
import json

scenes = json.load(open(meta / 'scenes.valid.json'))
scene_listeners = json.load(open(meta / 'scene_listeners.valid.json'))

scene_id = next(iter(scene_listeners))
listener_id = scene_listeners[scene_id][0]
print('scene   :', scene_id, scenes[scene_id])
print('listener:', listener_id)

one = root / 'metadata_one_scene'
one.mkdir(exist_ok=True)
json.dump({scene_id: scenes[scene_id]}, open(one / 'scenes.json', 'w'))
json.dump({scene_id: [listener_id]}, open(one / 'scene_listeners.json', 'w'))

scene   : S50001 {'music': 'anitrasdance_001', 'gain': 'gain_0236'}
listener: L5051


4. Run the baseline - Same as notebook 02 but on the valid set.

In [5]:
exp = Path('exp_valid_one_scene').resolve()
overrides = f'path.root={root} path.scenes_file={one}/scenes.json path.scene_listeners_file={one}/scene_listeners.json path.exp_folder={exp}'

!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/enhance.py {overrides}

[2026-11-01 02:59:44,694][__main__][INFO] - Loading model cadenzachallenge/ConvTasNet_Bassoon_NonCausal
[2026-11-01 02:59:45,119][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-11-01 02:59:45,120][huggingface_hub.utils._http][WARNING] - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-11-01 02:59:45,164][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/5828031665ea5ce9d8b05ec71e53c7eb68fe0761/config.json?%2Fcadenzachallenge%2FConvTasNet_Bassoon_NonCausal%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a049a1a444f751a3571e04e1d01697c68e99788e%22 "HTTP/1.1 200 OK"
[2026-11-01 02:59:45,213][httpx][INFO] - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cadenzachallenge/ConvTasNet_Bassoon_N

5. Compute HAAQI - Rebuilds the reference from the stems and then runs it through the listener's hearing-aid model, then compares it to our enhanced signal for each ear.

In [6]:
!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/evaluate.py {overrides}

[2026-11-01 03:04:24,467][__main__][INFO] - Evaluating from enhanced_signals directory
[2026-11-01 03:04:24,514][__main__][INFO] - [001/001] Evaluating S50001 for listener L5051
[2026-11-01 03:05:25,447][__main__][INFO] - Done!


6. Read the score - HAAQI  between 0 and 1, higher the better.

In [7]:
import csv

with open(exp / 'scores.csv') as f:
    rows = list(csv.DictReader(f))
for r in rows:
    print(f"scene {r['scene']} | listener {r['listener']} | song {r['song']}")
    print(f"left: {float(r['left_haaqi']):.4f}")
    print(f"right: {float(r['right_haaqi']):.4f}")
    print(f"avg: {float(r['avg_haaqi']):.4f}")

scene S50001 | listener L5051 | song anitrasdance_001
left: 0.4827
right: 0.4809
avg: 0.4818


## DONE
If a score printed, HAAQI is computed for the baseline output.